# Whole-Codebase Auditor — Colab driver

This notebook is a **driver, not the program**. All logic lives in the `wca` package,
so the same code runs here on a Colab GPU and on any machine via `wca scan <repo>`.
If you find yourself editing logic in a cell, it belongs in the repo instead.

**Runtime → Change runtime type → T4 GPU** before running.

Notes:
- We do **not** install `mamba-ssm` / `causal_conv1d`. They are optional; without them
  transformers uses the eager path, which is correct and simply slower. Installing
  them triggers a source build that fails on Colab regularly. Get correctness first.
- We do **not** reinstall `torch`. Colab ships a CUDA-matched build; replacing it
  breaks the runtime.

In [ ]:
#@title 1. Install the package
REPO = "https://github.com/quinyang/whole_codebase_auditor"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

# PEP 508 direct reference. The older "#egg=wca[gpu]" form is deprecated and on
# recent pip silently drops the [gpu] extra, which then fails at cell 5.
!pip install -q "wca[gpu] @ git+{REPO}@{BRANCH}"

import wca
print("wca", wca.__version__)

In [ ]:
#@title 2. Check the hardware before spending time on it
from wca.infer import describe_environment
print(describe_environment())

# Expect on free Colab:
#   device: Tesla T4 / compute dtype: torch.float16   <- fp16, NOT bf16 (T4 is Turing)
#   mamba_ssm: absent (eager path -- correct, slower)

In [ ]:
#@title 3. Persist artifacts to Drive (Colab sessions die at ~90min idle / 12h max)
import os

USE_DRIVE = True  #@param {type:"boolean"}
OUT_DIR = "/content/wca_runs"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/wca_runs"

os.makedirs(OUT_DIR, exist_ok=True)
print("artifacts ->", OUT_DIR)

In [ ]:
#@title 4. CPU stages: ingest → parse → graph → pack (no GPU needed, run this first)
TARGET = "pallets/flask"  #@param {type:"string"}
REF = "main"  #@param {type:"string"}
BUDGET = 24000  #@param {type:"integer"}

from wca.ingest import ingest
from wca.parse import parse_files, LanguageDispatcher
from wca.graph import build_graph
from wca.pack import pack

bundle = ingest(TARGET, REF)
print(bundle.summary())

parsed = parse_files(bundle.files, LanguageDispatcher())
print(parsed.summary())

graph = build_graph(parsed.files)
print(graph.summary())

packed = pack(parsed.files, graph, budget_tokens=BUDGET, repo_name=bundle.name)
print(packed.stats_line())

# Checkpoint: if the session dies during inference, this survives.
import os
stem = os.path.join(OUT_DIR, bundle.name.replace("/", "__"))
open(stem + ".pack.txt", "w").write(packed.text)
packed.write_manifest(stem + ".manifest.json")
print("checkpointed ->", stem + ".pack.txt")

In [ ]:
#@title 5. Load the model and audit (GPU; prefill dominates, expect minutes on a T4)
from wca.infer import MambaAuditor
from wca.findings import parse_findings, AuditReport

auditor = MambaAuditor()  # tiiuae/Falcon3-Mamba-7B-Instruct, 4-bit, dtype auto-selected

# Repack with the real tokenizer so the budget is exact rather than estimated.
packed = pack(parsed.files, graph, budget_tokens=BUDGET,
              tokenizer=auditor.tokenizer, repo_name=bundle.name)
print(packed.stats_line())

gen = auditor.generate(packed.text, max_new_tokens=1024)
print(gen.stats_line())

report = AuditReport(
    repo=f"{bundle.name}@{bundle.ref}",
    model=auditor.model_id,
    findings=parse_findings(gen.text, packed),
    pack_stats={"budget_tokens": packed.budget_tokens, "used_tokens": packed.used_tokens},
    gen_stats={"prompt_tokens": gen.prompt_tokens, "seconds": round(gen.total_seconds, 1)},
)
print(report.pretty())
report.save(stem + ".findings.json")

## Only if speed becomes the bottleneck

The eager path is correct but slow. Fast kernels require a **prebuilt wheel** matching
the exact `cu12x` / `torch2.x` / `cxx11abi` / `cp31x` combination of the current runtime.

**Do not run `pip install mamba-ssm`** — it is a source build and will hang or fail.

Find the matching asset at github.com/state-spaces/mamba/releases and
github.com/Dao-AILab/causal-conv1d/releases, then pin the exact URL below.
Colab bumps torch periodically and will silently break an unpinned wheel.

In [ ]:
# Print the exact combo you need, then pick the matching release asset.
import torch, sys
print("torch      ", torch.__version__)
print("cuda       ", torch.version.cuda)
print("cxx11abi   ", torch._C._GLIBCXX_USE_CXX11_ABI)
print("python     ", f"cp{sys.version_info.major}{sys.version_info.minor}")

# !pip install -q https://github.com/Dao-AILab/causal-conv1d/releases/download/<TAG>/<EXACT_WHEEL>.whl
# !pip install -q https://github.com/state-spaces/mamba/releases/download/<TAG>/<EXACT_WHEEL>.whl